# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROHITCRAFTSYT/flyrank-int/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 4 — CTR / Engagement Opportunity Scoring.**

I picked this lane because the starter data already shows the gap it depends on. Among pages with real exposure, click-through rate varies about six-fold *within a single position tier*: page-1 pages sit at 0.12% CTR at the 25th percentile and 0.75% at the 90th. If position alone decided clicks, that spread would be narrow. It isn't — so there is something left to explain, and that leftover is what a reviewer could act on.

Two things pushed me away from the alternatives. The refresh lane has more volume, but its headline baseline rule (`stale_visible_page`) fires on only **17 of 30,000 rows** here, and its starter label (`trend_direction == "down"`) is a bucket computed from the current window rather than a future outcome — the lane guide calls that out as a beginner proxy label (§5). The AI referral direction is thinner still: only **1,930 of 30,000 rows** have any AI sessions at all, which the guide says makes it EDA-only (§9).

The CTR lane also gives me a method I can defend rather than just a model I can fit: comparing a page only against others at the same position tier is a correction I have to argue for, and the guide names "comparing CTR across positions without adjusting for position" as the classic mistake in this lane (§8, Lane 4). I would rather spend seven weeks on a question where the honest method is the interesting part.

I can revise this until the end of Week 4.

In [1]:
# Setup + the numbers behind my lane choice.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Why not the other lanes -- the two numbers I cited above, checked here.
stale_visible = ((df.days_since_last_update >= 180) & (df.impressions_90d >= 500)).sum()
has_ai = (df.ai_sessions_90d > 0).sum()

print(f"rows in starter CSV: {len(df):,}")
print(f"refresh lane, 'stale_visible_page' rule fires on : {stale_visible:,} rows")
print(f"AI lane, rows with any AI session                : {has_ai:,} rows "
      f"({has_ai/len(df):.1%})")
print("-> both alternatives are thin here. Checking my own lane's universe next.")

rows in starter CSV: 30,000
refresh lane, 'stale_visible_page' rule fires on : 17 rows
AI lane, rows with any AI session                : 1,930 rows (6.4%)
-> both alternatives are thin here. Checking my own lane's universe next.


## 2. The question: decision, action, cost of a wrong call

**My search question:** *Among pages that are already visible in search, which ones are capturing fewer clicks than comparable pages at the same position — enough to be worth a human's review first?*

**Unit of analysis (my grain):** one pseudonymized content item (one row per `content_id`), filtered to pages that are actually visible. One row = one page a reviewer could open and edit. Not a client, not a day.

**The decision I improve.** A content team has more pages than review hours. The decision is not "is this page bad" — it is *"which page does an editor open on Monday morning?"* Today that ordering is made by a fixed rule or by whoever shouts loudest. I want to make it evidence-ranked.

**Who acts, and what they do.** A content editor. They open the page and take one concrete action: rewrite the title and meta description, fix an intent mismatch between what the page promises and what it delivers, or improve the snippet structure. If the evidence is weak, the action is "monitor" — which is a real action, not a cop-out.

**The output.** A ranked queue of review candidates, each carrying a score, a suggested action, and reason codes a human can inspect and overrule. Not a number handed down from a black box.

**What a wrong recommendation costs.** The costs are asymmetric, and that shapes the metric:

- **False positive** (I rank a page high, the editor finds nothing wrong): roughly an hour of editor time, plus a bit of trust in the queue. Trust is the expensive half — a queue that burns people twice stops being used at all.
- **False negative** (a genuinely under-performing page never surfaces): invisible, and therefore worse in the long run. The page keeps under-earning its impressions and nobody knows.
- **The dangerous wrong call**: recommending a rewrite for a page whose low CTR is *correct* — a page ranking for a query it shouldn't match, where clicks would be the wrong outcome anyway. Rewriting that page makes things worse, not neutral.

Because a reviewer only ever sees the top of the list, **precision@K is my metric** (K set by real review capacity, ~20–50 pages), not accuracy. Accuracy over 12,000 pages would mostly measure the boring majority.

**Why data or ML at all — and not an if-statement.** This is the question I have to answer honestly, because the starter baseline *is* an if-statement (`ctr < 0.5 AND impressions >= 500`), and it might be enough. My argument for going further: a flat CTR threshold is provably wrong here, because the expected CTR depends on position — a fixed cutoff quietly punishes every page ranking at position 9 and forgives every page at position 2. Correcting for that needs a comparison group per tier, and the residual after that correction depends on several signals at once (position, intent, content type, volume, freshness). That is the case where a model can earn its place. **But it has to earn it**: if a per-tier expected-CTR rule matches a trained model on precision@K, I will ship the rule and say so. "The simple thing won" is a real finding, not a failure.

**The frame in one paragraph** (per the framing skill):

> For a **content editor**, deciding **which visible page to review first**, we will build a **ranked queue with reason codes** from the **starter dataset and the warehouse daily facts**, scoring **how far a page's clicks fall below comparable pages at the same position**, measured by **precision@K at real review capacity**. A wrong call costs **an hour of editor time and, worse, trust in the queue** — and a missed page costs invisibly. A plain rule isn't enough because **a flat CTR cutoff ignores position, so it systematically mis-ranks by rank**. We will claim only **observed, directional, decision-support** results.

In [2]:
# The decision universe, and the capacity squeeze that makes ranking the whole point.

# Starter feature-prep rules (lane guide S5): impressions_90d > 0, content_age_days >= 90,
# deduplicated by content_id.
elig = df[(df.impressions_90d > 0) & (df.content_age_days >= 90)].drop_duplicates("content_id")

# GOTCHA (data skill S1): avg_position == 0 means "no position data", NOT rank zero.
# A page with no position cannot be compared to its tier, so it is out of scope for me.
no_position = (elig.avg_position == 0).sum()

# "Visible" = enough exposure to matter, and actually ranking where a click is possible.
visible = elig[(elig.impressions_90d >= 500) & (elig.avg_position > 0) & (elig.avg_position <= 20)]

print(f"eligible pages           : {len(elig):,}  across {elig.client_id.nunique()} clients")
print(f"  dropped: no position   : {no_position:,} rows (avg_position == 0 means 'no data')")
print(f"MY DECISION UNIVERSE     : {len(visible):,} visible pages ({len(visible)/len(elig):.0%} of eligible)")
print()

CAPACITY = 50  # pages one reviewer can genuinely work through in a week
print("The squeeze that makes this a ranking problem:")
print(f"  pages competing for review : {len(visible):,}")
print(f"  reviewer capacity per week : {CAPACITY}")
print(f"  so a reviewer ever sees    : top {CAPACITY/len(visible):.1%} of the universe")
print(f"  clearing it by random order: {len(visible)/CAPACITY:,.0f} weeks")
print()
print("-> The ordering IS the product. Only the top ~50 rows ever get acted on,")
print("   which is exactly why precision@K is the metric and accuracy is not.")

eligible pages           : 30,000  across 32 clients
  dropped: no position   : 1,205 rows (avg_position == 0 means 'no data')
MY DECISION UNIVERSE     : 12,023 visible pages (40% of eligible)

The squeeze that makes this a ranking problem:
  pages competing for review : 12,023
  reviewer capacity per week : 50
  so a reviewer ever sees    : top 0.4% of the universe
  clearing it by random order: 240 weeks

-> The ordering IS the product. Only the top ~50 rows ever get acted on,
   which is exactly why precision@K is the metric and accuracy is not.


## 3. Quick look at the data (2-3 real numbers)

Three numbers, each one load-bearing for the lane choice. The code below computes all three, and first verifies the scale of the `ctr` column instead of trusting the docs on it.

**Number 1 — CTR varies ~6x within a single position tier.** Page-1 pages run 0.12% CTR at the 25th percentile and 0.75% at the 90th, across 7,064 pages. Same tier, same rough rank, six-fold difference in clicks captured. If position were the whole story this spread would be tight. Something else is moving, and that is the thing worth seven weeks.

**Number 2 — position is a real but partial explanation.** Spearman correlation between `avg_position` and `ctr` is about −0.18: the right sign, clearly not the whole picture. Median CTR does fall as rank worsens (≈0.33% at position 4, ≈0.17% at position 10), which is why comparing across positions without adjusting is the mistake the guide warns about. But the correlation being far from −1 is precisely what leaves a residual to rank on.

**Number 3 — the opportunity is concentrated enough to act on.** About 3,031 visible pages sit below *half* their own tier's median CTR, and together they hold roughly 21.5M impressions in 90 days. That is a large enough pool that a reviewer's top-50 could plausibly be drawn from real cases rather than noise — and small enough that ranking within it actually matters.

**One honest caveat, up front.** The absolute CTR values here are far below what public search benchmarks report for page-1 results. I verify below that I am reading the column's scale correctly (aggregate clicks ÷ impressions agrees with the median), so this is not a units error on my part — but it does mean these rates are a property of this pseudonymized slice, not a claim about search in general. I will rank *within* this data and avoid comparing its levels to outside benchmarks.

In [3]:
# === Number 0: verify the ctr scale before quoting any of it ===
# The data skill says rate columns are x100 percentages (ctr = 0.76 means 0.76%).
# Don't take that on faith -- rebuild it from the raw counts and see if it agrees.
agg_ctr_pct = 100 * visible.clicks_90d.sum() / visible.impressions_90d.sum()
median_ctr_col = visible.ctr.median()
print("=== Number 0: is ctr really a x100 percentage? ===")
print(f"  aggregate 100 * clicks/impressions : {agg_ctr_pct:.3f}")
print(f"  median of the ctr column           : {median_ctr_col:.3f}")
print("  -> Same order of magnitude, so the ctr column is already in PERCENT units:")
print(f"     a value of {median_ctr_col:.2f} means {median_ctr_col:.2f}%, not {median_ctr_col*100:.0f}%.")
print("     (The aggregate sits a little above the median because clicks concentrate")
print("      in high-impression pages -- expected, and not a units problem.)")
print("     Reading it the other way would put every claim below off by 100x.")
print()

# === Number 1: CTR varies a lot WITHIN one position tier ===
print("=== Number 1: CTR spread inside a single position tier (units: %) ===")
spread = visible.groupby("position_tier")["ctr"].describe(percentiles=[.25, .5, .75, .9])
print(spread[["count", "25%", "50%", "75%", "90%"]].round(3).to_string())

p1 = visible.loc[visible.position_tier == "page_1", "ctr"]
print(f"\n  page_1: p90 CTR is {p1.quantile(.9)/p1.quantile(.25):.1f}x the p25 CTR "
      f"({p1.quantile(.9):.2f}% vs {p1.quantile(.25):.2f}%), across {len(p1):,} pages.")
print("  -> Same tier, six-fold difference. Position is not the whole story.")
print()

# === Number 2: position explains only part of CTR ===
print("=== Number 2: median CTR by rounded position (tiers >= 50 pages) ===")
byp = (visible.assign(pos=visible.avg_position.round())
              .groupby("pos").agg(pages=("ctr", "size"), median_ctr=("ctr", "median")))
print(byp[byp.pages >= 50].head(10).round(3).to_string())

corr = visible[["avg_position", "ctr"]].corr(method="spearman").iloc[0, 1]
print(f"\n  Spearman corr(avg_position, ctr) = {corr:.3f}")
print("  -> Real and negative (worse rank, fewer clicks) but far from -1.")
print("     So: never compare CTR across positions raw -- but the leftover after")
print("     adjusting for position is exactly what my queue will rank on.")
print()

# === Number 3: is the opportunity big enough to be worth 7 weeks? ===
print("=== Number 3: pages sitting below their OWN tier's median ===")
tier_median = visible.groupby("position_tier")["ctr"].transform("median")
below = visible[visible.ctr < tier_median]
far_below = visible[visible.ctr < 0.5 * tier_median]
print(f"  below their tier median CTR        : {len(below):,} pages")
print(f"  below HALF their tier median       : {len(far_below):,} pages")
print(f"  impressions_90d held by those pages: {far_below.impressions_90d.sum():,.0f}")
print(f"  spread across                      : {far_below.client_id.nunique()} clients")
print("  -> Big enough that a top-50 queue can be drawn from real cases, not noise.")
print("     NOTE: this is a DESCRIPTION of the current window, not yet a prediction.")
print("     Section 4 says why that distinction decides my next task.")

=== Number 0: is ctr really a x100 percentage? ===
  aggregate 100 * clicks/impressions : 0.358
  median of the ctr column           : 0.210
  -> Same order of magnitude, so the ctr column is already in PERCENT units:
     a value of 0.21 means 0.21%, not 21%.
     (The aggregate sits a little above the median because clicks concentrate
      in high-impression pages -- expected, and not a units problem.)
     Reading it the other way would put every claim below off by 100x.

=== Number 1: CTR spread inside a single position tier (units: %) ===
                count    25%    50%    75%    90%
position_tier                                    
page_1         7064.0  0.120  0.240  0.440  0.750
page_3_5         16.0  0.057  0.155  0.245  0.440
striking       4485.0  0.080  0.170  0.340  0.610
top_3           458.0  0.080  0.200  0.490  0.863

  page_1: p90 CTR is 6.2x the p25 CTR (0.75% vs 0.12%), across 7,064 pages.
  -> Same tier, six-fold difference. Position is not the whole story.

=

## 4. Careful words: what I can and can't claim

**What I will be able to say (observed / directional / decision-support):**

- *"We observed that CTR varies about six-fold within a single position tier."* Measured, in this slice. Safe.
- *"Pages ranked lower show directionally lower median CTR (Spearman ≈ −0.18)."* Directional, with the effect size attached and the sign stated. Safe.
- *"This queue ranks pages that under-capture clicks relative to comparable pages at the same position, so a reviewer's limited hours go to the most promising candidates first."* Decision-support — a claim about prioritization, not about outcomes. Safe.
- *"On a client-holdout split, the ranked queue placed N of its top 50 on pages that later [observed outcome], versus M for the rule baseline."* Safe **only once** the split and the label are built and audited.

**What I will never claim, no matter how good the numbers look:**

- **Not causal.** I will never say a rewritten title *caused* a CTR recovery. Proving that needs an experiment or a causal design, and this data has neither (lane guide §6, §14). The most I can say is that a page looked like a promising candidate and something changed afterward.
- **Not about Google.** I will never say I found a ranking factor or reverse-engineered the algorithm. I observe correlations in one pseudonymized slice of 32 clients.
- **Not a guarantee.** A high score means "worth a human's hour," never "this will improve if edited."
- **Not generalizable levels.** As flagged in section 3, the absolute CTR rates here are far below public benchmarks. I will rank *within* this data and never quote its CTR levels as a fact about search.
- **Nothing identifying.** No client names, domains, URLs, titles, or raw queries — only pseudonymous IDs and aggregates (§14).

**The trap I have already spotted in my own lane — and what it makes my next task.**

My whole lane rests on a "CTR gap," and the gap is computed *from* `ctr`, which is `clicks_90d / impressions_90d`. So `ctr` and `clicks_90d` can never be features for a model whose target is derived from the gap — that would be a circular result, the exact trap the lane guide describes in §4. `trend_direction` and `trend_pct` are leak-listed too: they are the starter label's own ancestors (data skill §1).

Worse, and more usefully: everything I computed above is a **description of one 90-day window, not a prediction of anything.** A page whose CTR sat below its tier last quarter is a page that *has* under-performed — that is not the same as a page that *will* benefit from review. The starter CSV cannot fix this; its `last_30d`/`prev_30d` columns live inside the same snapshot as `ctr`, so no honest forward window exists in this file. The code cell below shows that concretely.

That gives me a clear next task rather than a vague one: **go to the warehouse daily facts and build a real forward window** — features from a prior window, outcome measured in a strictly later one, with the per-client history checks the data skill demands (`gsc_data_start`, and `ga4_data_available IS TRUE`, since NULL is neither true nor false). Until that label exists, my strongest honest claim is a *descriptive* ranking: "these pages under-captured clicks relative to their tier last quarter." That is genuinely useful to a reviewer, and I would rather ship it labelled honestly than dress it up as prediction.

In [4]:
# Guardrails, written down now so Week 3's leakage check has something to test against.

print("=== My leak list for this lane (never features) ===")
LEAK_LIST = {
    "ctr":             "the gap IS derived from it -- circular (lane guide S4)",
    "clicks_90d":      "ctr = clicks_90d / impressions_90d, so this rebuilds the target",
    "trend_direction": "ancestor of the starter label; computed from trend_pct (data skill S1)",
    "trend_pct":       "same family -- never a feature",
    "content_id":      "pseudonym: group/split key only, never a feature",
    "client_id":       "pseudonym: use for GROUPED train/test splits, never a feature",
}
for col, why in LEAK_LIST.items():
    print(f"  - {col:<16} {why}")
print()

# Does the starter CSV contain any honest forward window? Check, don't assume.
print("=== Can I build a forward-looking label from this file? ===")
window_cols = [c for c in df.columns if "last_30" in c or "prev_30" in c]
print(f"  window-ish columns present: {window_cols}")
print("  ...but all of them are cut from the SAME 90d snapshot that ctr comes from.")
print("  A 'prev 30 vs last 30' comparison inside one snapshot describes the past;")
print("  it is not a future outcome measured after a decision point.")
print("  -> VERDICT: no honest forward label exists in the starter CSV.")
print("     Next task: build one from the warehouse daily facts (prior window -> later window).")
print()

# The panel trap I'll hit the moment I touch the warehouse -- noted now so I don't forget.
print("=== Warehouse checks queued for next week (data skill S2) ===")
print("  - filter with `ga4_data_available IS TRUE` (NOT `= FALSE`, NOT `NOT flag`):")
print("    millions of rows are NULL, and NULL is neither -- the wrong form miscounts silently.")
print("  - check dim_clients.gsc_data_start per client before fixing any date window;")
print("    history depth varies wildly, so a global calendar window would be wrong.")
print("  - develop on ONE middle month (e.g. month=2026-03); the _sample table is the LAST")
print("    month, which for a forward-looking label is the answer key.")
print()

# Public-safety check on this notebook itself (lane guide S14).
print("=== Public-safety check on this notebook ===")
UNSAFE = ["url", "domain", "title", "query", "keyword_text", "client_name"]
shown = set()
for frame_name, frame in [("df", df), ("visible", visible)]:
    for col in frame.columns:
        if any(u in col.lower() for u in UNSAFE):
            shown.add(col)
print(f"  identifying-looking columns in the data : {sorted(shown) or 'none'}")
print(f"  identifying values printed above        : none -- only counts, quantiles, and")
print(f"                                            aggregates across {visible.client_id.nunique()} pseudonymous clients")

=== My leak list for this lane (never features) ===
  - ctr              the gap IS derived from it -- circular (lane guide S4)
  - clicks_90d       ctr = clicks_90d / impressions_90d, so this rebuilds the target
  - trend_direction  ancestor of the starter label; computed from trend_pct (data skill S1)
  - trend_pct        same family -- never a feature
  - content_id       pseudonym: group/split key only, never a feature
  - client_id        pseudonym: use for GROUPED train/test splits, never a feature

=== Can I build a forward-looking label from this file? ===
  window-ish columns present: ['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']
  ...but all of them are cut from the SAME 90d snapshot that ctr comes from.
  A 'prev 30 vs last 30' comparison inside one snapshot describes the past;
  it is not a future outcome measured after a decision point.
  -> VERDICT: no honest forward label exists in the sta

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — checked in code, not just by eye: the starter CSV contains no URL/domain/title/query columns, and nothing below the aggregate level is printed
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Where I actually stand, in one line:** lane provisionally set to CTR/Engagement Opportunity Scoring, backed by three measured numbers from the starter data — and I have already found the thing that will bite me (no honest forward window exists in the starter CSV), which makes Week 3's job concrete instead of vague.

**Open questions I am carrying forward, not hiding:**

1. **Are these CTR levels trustworthy in absolute terms?** They sit far below public page-1 benchmarks. I verified the units are right, so it is a property of this slice — but I want to understand *why* before I lean on it. Possible: pseudonymization scaling, or a client mix that skews to high-impression/low-intent queries. Either way, ranking within the data is safe; quoting the levels outward is not.
2. **Is `position_tier` the right comparison group?** It is coarse, and `top_3` shows a *lower* median CTR than `page_1` (0.20% vs 0.24%) — the opposite of what rank alone predicts, on only 458 pages. That inversion is either a small-sample artefact or a real signal about which queries reach the top 3. Worth a proper look in the signal audit; it may push me toward finer position bands or a modelled expected-CTR curve instead of tiers.
3. **What is the honest outcome to measure?** "CTR later improved" is observable but confounded by position drift — a page can gain clicks purely by ranking better, with no editorial merit. The label needs to hold position roughly constant, which is a design problem for the data contract, not something to hand-wave.